In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.14 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# CeCoAl₃ on POLARIS

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit CeCoAl₃ to time-of-flight neutron powder data from POLARIS at ISIS: structure and profile,
with a 24-term Chebyshev background.

It builds the catalogue case `pd-neut-tof_cecoal-polaris` step by step: the same project the
command line fits and the app's Examples open.

## Import the library

In [2]:
import easydiffraction as edi

## Define the project

A new project is empty: it holds no structure and no experiment until you add them.

In [3]:
project = edi.Project(name='pd-neut-tof_cecoal-polaris')

## Define the structure

Load the structure from EasyDiffraction's collection. Its CIF gives the space group, the cell and
the atom sites.

In [4]:
path = edi.download_data('pd-neut-tof_cecoal-polaris-cecoal', destination='data')
project.structures.add_from_cif_path(path)
structure = project.structures['cecoal']

Data 'pd-neut-tof_cecoal-polaris-cecoal' is at data/pd-neut-tof_cecoal-polaris-cecoal.edi


In [5]:
project.display.structure('cecoal')

View(html=<226,817 characters>, height=660)

## Define the experiment

Create the experiment from the measured data in EasyDiffraction's collection, then describe the
instrument, the peak profile, the background and the structure it links.

In [6]:
data_path = edi.download_data('pd-neut-tof_cecoal-polaris-polaris-data', destination='data')
project.experiments.add_from_data_path(
    name='polaris',
    data_path=data_path,
    sample_form='powder',
    beam_mode='time-of-flight',
    radiation_probe='neutron',
    scattering_type='bragg',
)
experiment = project.experiments['polaris']
experiment.neutron_scattering_length = 'sears1992'
experiment.peak.type = 'tof-jorgensen-von-dreele'
experiment.absorption.type = 'cylinder'
experiment.background.type = 'chebyshev'
experiment.instrument.name = 'POLARIS'
experiment.instrument.facility = 'ISIS'
experiment.peak.cutoff_fwhm = 4.2
experiment.peak.rise_alpha_1 = 0.2971
experiment.peak.decay_beta_0 = 0.043085
experiment.peak.decay_beta_1 = 0.002867
experiment.peak.broad_gauss_sigma_1 = 40
experiment.peak.broad_lorentz_gamma_1 = 5
experiment.instrument.setup_twotheta_bank = 145.0
experiment.instrument.calib_d_to_tof_linear = 6176.03076
experiment.instrument.calib_d_to_tof_quadratic = -1.92194
experiment.background_x_min = 3001.5891
experiment.background_x_max = 19004.6582
experiment.excluded_regions.create(start=2000.1, end=3000.0)
experiment.excluded_regions.create(start=19000.0, end=20000.0)
experiment.linked_structures.create(structure_id='cecoal', scale=20000)
terms = []
for coef in (25000.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0):
    term = edi.PolynomialTerm()
    term.order = len(terms)
    term.coef = coef
    terms.append(term)
experiment.background_terms = terms

Data 'pd-neut-tof_cecoal-polaris-polaris-data' is at data/pd-neut-tof_cecoal-polaris-polaris.xye


In [7]:
project.display.pattern('polaris')

View(html=<600,393 characters>, height=480)

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time.

### Scale, time-of-flight calibration and cell

The scale, the DIFC offset and its quadratic term, and the three cell lengths put the peaks in
place and at the right height.

In [8]:
experiment.linked_structures['cecoal'].scale.free = True
experiment.instrument.calib_d_to_tof_offset.free = True
experiment.instrument.calib_d_to_tof_quadratic.free = True
structure.cell.length_a.free = True
structure.cell.length_b.free = True
structure.cell.length_c.free = True

In [9]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,cecoal,cell,,length_a,7.65,0,0,30,Å
2,cecoal,cell,,length_b,4.05,0,0,30,Å
3,cecoal,cell,,length_c,6.91,0,0,30,Å
4,polaris,instrument,,calib_d_to_tof_offset,0,0,-inf,inf,µs
5,polaris,instrument,,calib_d_to_tof_quadratic,-1.92194,0,-inf,inf,µs/Å²
6,polaris,linked_structure,cecoal,scale,2.00×10⁴,0,-inf,inf,


In [10]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 6 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       11.38        84.57
      1      0.528      4.13        11.13   86.8% ↓
      2      0.654      4.13        11.13   
  done · converged · 2 iters · reduced χ² 11.13 · Rwp 4.13 % · 0.74 s


View(html=<88,714 characters>, height=376)

In [11]:
project.display.pattern('polaris')

View(html=<598,554 characters>, height=480)

### Background

The first twelve polynomial terms (orders 0 to 11) follow the level between the peaks; the
higher ones stay at zero.

In [12]:
for term in list(experiment.background_terms)[:12]:
    term.coef.free = True

In [13]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 18 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        4.13        11.17
      1      0.645      2.31         3.51   68.5% ↓
  done · converged · 1 iters · reduced χ² 3.51 · Rwp 2.31 % · 0.73 s


View(html=<101,438 characters>, height=376)

In [14]:
project.display.pattern('polaris')

View(html=<618,097 characters>, height=480)

### Peak widths

In [15]:
experiment.peak.broad_gauss_sigma_1.free = True
experiment.peak.broad_lorentz_gamma_1.free = True

In [16]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 20 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.31         3.51
      1      0.653      2.31         3.51   0.2% ↓
  done · converged · 1 iters · reduced χ² 3.51 · Rwp 2.31 % · 0.74 s


View(html=<107,368 characters>, height=376)

In [17]:
project.display.pattern('polaris')

View(html=<618,033 characters>, height=480)

### Atom positions

The coordinates the symmetry leaves free.

In [18]:
for site in structure.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [19]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 25 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.31         3.51
      1      0.600      2.14         3.00   14.5% ↓
      2      0.732      2.14         3.00   
  done · converged · 2 iters · reduced χ² 3.00 · Rwp 2.14 % · 0.82 s


View(html=<114,356 characters>, height=376)

In [20]:
project.display.pattern('polaris')

View(html=<618,245 characters>, height=480)

### Displacements

In [21]:
for site in structure.atom_sites:
    site.adp_iso.free = True

In [22]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 30 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.14         3.01
      1      0.669      2.10         2.89   3.7% ↓
  done · converged · 1 iters · reduced χ² 2.89 · Rwp 2.10 % · 0.76 s


View(html=<120,165 characters>, height=376)

In [23]:
project.display.pattern('polaris')

View(html=<618,093 characters>, height=480)

### Absorption

Absorption weakens the intensities with time of flight, as the displacements do with angle, so
it comes after them, on its own.

In [24]:
experiment.absorption.abscor1.free = True

In [25]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 31 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.10         2.90
      1      0.608      2.08         2.86   1.3% ↓
      2      0.740      2.08         2.86   
  done · converged · 2 iters · reduced χ² 2.86 · Rwp 2.08 % · 0.83 s


View(html=<121,336 characters>, height=376)

In [26]:
project.display.pattern('polaris')

View(html=<618,275 characters>, height=480)

### Al3 occupancy

Last, Al3's occupancy, which moves with its displacement.

In [27]:
structure.atom_sites['Al3'].occupancy.free = True

In [28]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 32 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.08         2.86
      1      0.680      2.08         2.85   0.2% ↓
  done · converged · 1 iters · reduced χ² 2.85 · Rwp 2.08 % · 0.77 s


View(html=<122,662 characters>, height=376)

In [29]:
project.display.pattern('polaris')

View(html=<618,385 characters>, height=480)

In [30]:
project.display.fit.correlations()

,1,2,3,4,5
1 cecoal.cell.length_b,,,,,
2 cecoal.cell.length_c,0.94,,,,
3 polaris.instrument.calib_d_to_tof_quadratic,-0.96,-0.95,,,
4 polaris.background.1.coef,0.00,0.00,-0.00,,
5 polaris.background.2.coef,0.00,0.00,-0.00,0.97,


In [31]:
project.display.structure('cecoal')

View(html=<231,051 characters>, height=660)

## Report

The HTML report draws the measured, calculated and difference patterns with the reflection
positions, and lists the structures, the experiments and the fit. Open it in a browser.

In [32]:
project.report.save_html('pd-neut-tof_cecoal-polaris.html')

Saving report to 'pd-neut-tof_cecoal-polaris.html'


## Save the project

In [33]:
project.save_as('pd-neut-tof_cecoal-polaris')

Saving project to 'pd-neut-tof_cecoal-polaris'
